# Radar-Based Behaviour Simulator
## Version 1 — Implementation

## 0 · Imports

In [ ]:
import numpy as np
import pandas as pd
import random
from dataclasses import dataclass
from typing import Dict, List
from scipy import stats
import plotly.graph_objects as go
import plotly.subplots as sp

np.random.seed(42)
random.seed(42)

---
## 1 · Feature Set

One aggregate feature vector per resident per day.
Distribution families : durations → Gamma; counts → Negative-Binomial;
fractions → Beta; breathing rate → Normal; rare events → Zero-inflated Gamma.

In [2]:
FEATURES = {
    # Daytime posture and dwell
    'chair_duration':          {'family': 'gamma',              'unit': 'min',    'bounds': (0, 720)},
    'daytime_bed_duration':    {'family': 'gamma',              'unit': 'min',    'bounds': (0, 480)},
    'inactive_duration':       {'family': 'gamma',              'unit': 'min',    'bounds': (0, 960)},
    # Daytime activity and motion
    'active_duration':         {'family': 'gamma',              'unit': 'min',    'bounds': (0, 480)},
    'large_motion_count':      {'family': 'negative_binomial',  'unit': 'count',  'bounds': (0, 200)},
    'small_motion_count':      {'family': 'negative_binomial',  'unit': 'count',  'bounds': (0, 500)},
    # Nighttime sleep window
    'bed_occupancy_duration':  {'family': 'gamma',              'unit': 'min',    'bounds': (0, 600)},
    'sleep_window':            {'family': 'log_normal',         'unit': 'hours',  'bounds': (0, 12)},
    # Nighttime motion and fragmentation
    'small_motions_in_bed':    {'family': 'negative_binomial',  'unit': 'count',  'bounds': (0, 100)},
    'large_motions_night':     {'family': 'negative_binomial',  'unit': 'count',  'bounds': (0, 50)},
    'bed_exit_events':         {'family': 'negative_binomial',  'unit': 'count',  'bounds': (0, 15)},
    'sleep_fragmentation_idx': {'family': 'beta',               'unit': 'index',  'bounds': (0, 1)},
    # Nighttime respiration
    'breathing_rate_mean':     {'family': 'normal',             'unit': 'br/min', 'bounds': (6, 30)},
    'breathing_rate_var':      {'family': 'gamma',              'unit': 'br/min', 'bounds': (0, 10)},
    'apnea_count':             {'family': 'zero_inflated_nb',   'unit': 'count',  'bounds': (0, 100)},
    'apnea_duration':          {'family': 'zero_inflated_gamma','unit': 'min',    'bounds': (0, 300)},
    'cheyne_stokes_duration':  {'family': 'zero_inflated_gamma','unit': 'min',    'bounds': (0, 240)},
    # Bathroom
    'night_bathroom_visits':   {'family': 'negative_binomial',  'unit': 'count',  'bounds': (0, 15)},
}

print(f'Total features: {len(FEATURES)}')
pd.DataFrame(FEATURES).T

Total features: 18


,family,unit,bounds
chair_duration,gamma,min,"(0, 720)"
daytime_bed_duration,gamma,min,"(0, 480)"
inactive_duration,gamma,min,"(0, 960)"
active_duration,gamma,min,"(0, 480)"
large_motion_count,negative_binomial,count,"(0, 200)"
small_motion_count,negative_binomial,count,"(0, 500)"
bed_occupancy_duration,gamma,min,"(0, 600)"
sleep_window,log_normal,hours,"(0, 12)"
small_motions_in_bed,negative_binomial,count,"(0, 100)"
large_motions_night,negative_binomial,count,"(0, 50)"


---
## 2 · Parameter Library — Clinical Priors
Encodes the direction and magnitude of each disease's effect on each feature.

- `direction`: +1 increase / -1 decrease / 0 no effect
- `confounded`: True = feature shared across conditions
- `delta_mild / delta_severe`: additive shift from personal baseline

In [3]:
DISEASE_PARAMS = {

    # ── Reduced mobility ─────────────────────────────────────────────────
    'reduced_mobility': {
        'active_duration':        {'direction': -1, 'confounded': True,  'delta_mild': -15, 'delta_severe': -40},
        'large_motion_count':     {'direction': -1, 'confounded': True,  'delta_mild': -5,  'delta_severe': -15},
        'inactive_duration':      {'direction': +1, 'confounded': True,  'delta_mild': +20, 'delta_severe': +60},
        'chair_duration':         {'direction': +1, 'confounded': True,  'delta_mild': +15, 'delta_severe': +45},
        'daytime_bed_duration':   {'direction': +1, 'confounded': True,  'delta_mild': +10, 'delta_severe': +30},
        'breathing_rate_mean':    {'direction':  0, 'confounded': False, 'delta_mild':   0, 'delta_severe':   0},
        'cheyne_stokes_duration': {'direction':  0, 'confounded': False, 'delta_mild':   0, 'delta_severe':   0},
    },

    # ── Heart failure ────────────────────────────────────────────────────
    'heart_failure': {
        'cheyne_stokes_duration':  {'direction': +1, 'confounded': False, 'delta_mild': +5,   'delta_severe': +25},
        'night_bathroom_visits':   {'direction': +1, 'confounded': True,  'delta_mild': +2,   'delta_severe': +6},
        'bed_exit_events':         {'direction': +1, 'confounded': True,  'delta_mild': +2,   'delta_severe': +5},
        'sleep_fragmentation_idx': {'direction': +1, 'confounded': True,  'delta_mild': +0.1, 'delta_severe': +0.3},
        'breathing_rate_mean':     {'direction': +1, 'confounded': True,  'delta_mild': +1.5, 'delta_severe': +4.0},
        'active_duration':         {'direction': -1, 'confounded': True,  'delta_mild': -15,  'delta_severe': -40},
    },

    # ── COPD ─────────────────────────────────────────────────────────────
    'COPD': {
        'active_duration':         {'direction': -1, 'confounded': True,  'delta_mild': -15,  'delta_severe': -40},
        'large_motion_count':      {'direction': -1, 'confounded': True,  'delta_mild': -5,   'delta_severe': -15},
        'sleep_fragmentation_idx': {'direction': +1, 'confounded': True,  'delta_mild': +0.1, 'delta_severe': +0.3},
        'small_motions_in_bed':    {'direction': +1, 'confounded': True,  'delta_mild': +3,   'delta_severe': +10},
        'breathing_rate_mean':     {'direction': +1, 'confounded': True,  'delta_mild': +1.0, 'delta_severe': +3.5},
        'chair_duration':          {'direction': +1, 'confounded': True,  'delta_mild': +15,  'delta_severe': +45},
        'inactive_duration':       {'direction': +1, 'confounded': True,  'delta_mild': +20,  'delta_severe': +50},
        'daytime_bed_duration':    {'direction': +1, 'confounded': True,  'delta_mild': +10,  'delta_severe': +30},
        'cheyne_stokes_duration':  {'direction':  0, 'confounded': False, 'delta_mild':   0,  'delta_severe':   0},
    },

    # ── Sleep-disordered breathing ───────────────────────────────────────
    'sleep_disordered_breathing': {
        'apnea_count':             {'direction': +1, 'confounded': False, 'delta_mild': +5,   'delta_severe': +20},
        'apnea_duration':          {'direction': +1, 'confounded': False, 'delta_mild': +10,  'delta_severe': +40},
        'small_motions_in_bed':    {'direction': +1, 'confounded': True,  'delta_mild': +3,   'delta_severe': +10},
        'sleep_fragmentation_idx': {'direction': +1, 'confounded': True,  'delta_mild': +0.1, 'delta_severe': +0.3},
        'night_bathroom_visits':   {'direction': +1, 'confounded': True,  'delta_mild': +2,   'delta_severe': +5},
        'bed_exit_events':         {'direction': +1, 'confounded': True,  'delta_mild': +2,   'delta_severe': +4},
        'sleep_window':            {'direction': -1, 'confounded': False, 'delta_mild': -0.5, 'delta_severe': -1.5},
        'active_duration':         {'direction': -1, 'confounded': True,  'delta_mild': -10,  'delta_severe': -25},
    },
}

for disease, feats in DISEASE_PARAMS.items():
    print(f'{disease}: {len(feats)} features configured')

reduced_mobility: 7 features configured
heart_failure: 6 features configured
COPD: 9 features configured
sleep_disordered_breathing: 8 features configured


---
## 3 · Resident Profile Generator

Takes a configuration as input: comorbidities, severity levels, medications, simulation length.

In [4]:
@dataclass
class ResidentProfile:
    resident_id:     str
    conditions:      Dict[str, str]
    simulation_days: int = 180

    def summary(self):
        print(f"Resident  : {self.resident_id}")
        print(f"Conditions: {self.conditions}")
        print(f"Sim. days : {self.simulation_days}")


RESIDENT_CONFIGS = [
    ResidentProfile(
        resident_id="R001_mobility",
        conditions={"reduced_mobility": "moderate"},
    ),
    ResidentProfile(
        resident_id="R002_heart_failure",
        conditions={"heart_failure": "moderate"},
    ),
    ResidentProfile(
        resident_id="R003_COPD",
        conditions={"COPD": "moderate"},
    ),
    ResidentProfile(
        resident_id="R004_SDB",
        conditions={"sleep_disordered_breathing": "moderate"},
    ),
    ResidentProfile(
        resident_id="R005_comorbid",
        conditions={"heart_failure": "mild", "COPD": "moderate"},
    ),
]

for r in RESIDENT_CONFIGS:
    r.summary()
    print()


Resident  : R001_mobility
Conditions: {'reduced_mobility': 'moderate'}
Sim. days : 180

Resident  : R002_heart_failure
Conditions: {'heart_failure': 'moderate'}
Sim. days : 180

Resident  : R003_COPD
Conditions: {'COPD': 'moderate'}
Sim. days : 180

Resident  : R004_SDB
Conditions: {'sleep_disordered_breathing': 'moderate'}
Sim. days : 180

Resident  : R005_comorbid
Conditions: {'heart_failure': 'mild', 'COPD': 'moderate'}
Sim. days : 180



---
## 4 · Personal Baseline Model

Each resident's stable healthy pattern:
- `theta` — baseline level (median)
- `phi` — dispersion (MAD)

Population priors used as starting point before calibration.

In [5]:
POPULATION_PRIORS = {
    'chair_duration':          {'theta': 180,  'phi': 45},
    'daytime_bed_duration':    {'theta': 60,   'phi': 30},
    'inactive_duration':       {'theta': 300,  'phi': 60},
    'active_duration':         {'theta': 90,   'phi': 20},
    'large_motion_count':      {'theta': 12,   'phi': 4},
    'small_motion_count':      {'theta': 30,   'phi': 10},
    'bed_occupancy_duration':  {'theta': 480,  'phi': 60},
    'sleep_window':            {'theta': 7.5,  'phi': 0.8},
    'small_motions_in_bed':    {'theta': 8,    'phi': 3},
    'large_motions_night':     {'theta': 2,    'phi': 1},
    'bed_exit_events':         {'theta': 1,    'phi': 1},
    'sleep_fragmentation_idx': {'theta': 0.15, 'phi': 0.05},
    'breathing_rate_mean':     {'theta': 14.5, 'phi': 1.5},
    'breathing_rate_var':      {'theta': 1.2,  'phi': 0.4},
    'apnea_count':             {'theta': 0,    'phi': 0},
    'apnea_duration':          {'theta': 0,    'phi': 0},
    'cheyne_stokes_duration':  {'theta': 0,    'phi': 0},
    'night_bathroom_visits':   {'theta': 1,    'phi': 1},
}

def build_baseline():
    baseline = {}
    for feat, vals in POPULATION_PRIORS.items():
        baseline[feat] = {
            'theta':  vals['theta'],
            'phi':    vals['phi'],
            'family': FEATURES[feat]['family'] if feat in FEATURES else 'normal',
            'bounds': FEATURES[feat]['bounds'] if feat in FEATURES else (0, None),
        }
    return baseline

baseline = build_baseline()
print('Baseline ready.')
pd.DataFrame(baseline).T

Baseline ready.


,theta,phi,family,bounds
chair_duration,180,45,gamma,"(0, 720)"
daytime_bed_duration,60,30,gamma,"(0, 480)"
inactive_duration,300,60,gamma,"(0, 960)"
active_duration,90,20,gamma,"(0, 480)"
large_motion_count,12,4,negative_binomial,"(0, 200)"
small_motion_count,30,10,negative_binomial,"(0, 500)"
bed_occupancy_duration,480,60,gamma,"(0, 600)"
sleep_window,7.5,0.8,log_normal,"(0, 12)"
small_motions_in_bed,8,3,negative_binomial,"(0, 100)"
large_motions_night,2,1,negative_binomial,"(0, 50)"


---
## 5 · Health-State Progression
Latent states evolve in **logit space** bounded in (0, 1).
### Update rule
$$
\text{logit}(z_d) = B \cdot \text{logit}(z_{d-1}) + \text{drift} + \text{jump}_d + \text{recovery}_d + \varepsilon_d
$$
| Term | Role |
|---|---|
| $B \cdot \text{logit}(z_{d-1})$ | State persistence today depends strongly on yesterday |
| drift | Progressive decline |
| jump$_d$ | Sudden event |
| recovery$_d$ | Exponential decay back to baseline after an event |
| $\varepsilon_d$ | Day-to-day noise |
### Hazard model for acute events
Rather than a fixed jump probability each day, the event rate depends on the
current latent severity $z_d$:
$$
\lambda_d = \lambda_0 \cdot \exp(\alpha \cdot z_{d-1})
$$

- $\lambda_0$ - baseline event rate (events per day when healthy)  
- $\alpha$ - sensitivity of risk to current severity  
- Higher $z$ → higher $\lambda_d$ → more likely to trigger a jump
For **COPD**, a seasonal multiplier is added :
$$
\lambda_d^{\text{COPD}} = \lambda_0 \cdot \exp(\alpha \cdot z_{d-1}) \cdot \underbrace{(1 + \beta \cdot \cos(2\pi d / 365))}_{\text{seasonal factor}}
$$
Three latent states: **M** (mobility), **CP** (cardiopulmonary), **SDB** (sleep-disordered breathing)


In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))
def logit_inv(x):
    return 1 / (1 + np.exp(-x))
def hazard_rate(z, lambda0, alpha, day=0, seasonal=False, beta=0.3):
    """
    Compute the daily event rate using the hazard model.

    lambda_d = lambda0 * exp(alpha * z)        (base hazard)
             * (1 + beta * cos(2*pi*d/365))    (seasonal factor, COPD only)

    Parameters
    ----------
    z        : current latent severity (0-1)
    lambda0  : baseline event rate when healthy
    alpha    : sensitivity of risk to severity
    day      : simulation day (for seasonal effect)
    seasonal : True for COPD — winter exacerbation clustering
    beta     : seasonal amplitude (0 = no effect, 1 = strong)
    """
    rate = lambda0 * np.exp(alpha * z)
    if seasonal:
        rate *= (1 + beta * np.cos(2 * np.pi * day / 365))
    return rate


def generate_latent_trajectory(
    n_days,
    z0=0.05,
    B=0.97,
    drift=0.001,
    lambda0=0.02,      # baseline event rate (replaces fixed jump_rate)
    alpha=2.0,         # hazard sensitivity to severity
    jump_size=0.4,
    recovery=0.15,
    noise_sd=0.05,
    seasonal=False,    # True for COPD
):
    z = np.zeros(n_days)
    z[0] = z0
    in_recovery = False
    recovery_signal = 0.0
    lambda_log = []    # track daily hazard rate for inspection

    for d in range(1, n_days):
        # ── Hazard model: state-dependent event rate ───────────────────────
        lambda_d = hazard_rate(z[d-1], lambda0, alpha, day=d, seasonal=seasonal)
        lambda_log.append(lambda_d)

        # ── Acute event: Poisson draw with dynamic rate ────────────────────
        jump = 0.0
        if np.random.poisson(lambda_d) > 0:
            jump = jump_size
            in_recovery = True
            recovery_signal = jump_size

        # ── Recovery: exponential decay ────────────────────────────────────
        rec = 0.0
        if in_recovery:
            recovery_signal *= (1 - recovery)
            rec = -recovery_signal
            if recovery_signal < 0.01:
                in_recovery = False

        # ── State update in logit space ────────────────────────────────────
        logit_z = B * logit(z[d-1]) + drift + jump + rec + np.random.normal(0, noise_sd)
        z[d] = logit_inv(logit_z)

    return z, lambda_log


# Generate the 3 latent trajectories
n_days = 180

trajectory_M,   hazard_M   = generate_latent_trajectory(n_days, drift=0.001, lambda0=0.02, alpha=2.0)
trajectory_CP,  hazard_CP  = generate_latent_trajectory(n_days, drift=0.002, lambda0=0.02, alpha=2.0)
trajectory_SDB, hazard_SDB = generate_latent_trajectory(n_days, drift=0.0,   lambda0=0.01, alpha=1.5)

traj_df = pd.DataFrame({
    'day':        range(n_days),
    'M_mobility': trajectory_M,
    'CP_cardio':  trajectory_CP,
    'SDB_sleep':  trajectory_SDB,
})

print("Trajectory stats:")
print(traj_df[["M_mobility","CP_cardio","SDB_sleep"]].describe().round(3))


Trajectory stats:


,M_mobility,CP_cardio,SDB_sleep
count,180.000,180.000,180.000
mean,0.417,0.167,0.252
std,0.172,0.100,0.137
min,0.050,0.035,0.036
25%,0.296,0.080,0.137
50%,0.476,0.148,0.237
75%,0.570,0.232,0.349
max,0.614,0.409,0.512


### Visualize latent trajectories + hazard rate

In [7]:
fig = sp.make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "Latent severity trajectories z(t) — 180 days",
        "Daily hazard rate λ(t) — state-dependent event probability",
    ),
    vertical_spacing=0.12,
)

# Row 1 — severity trajectories
for col, color, name in [
    ("M_mobility", "#E67E22", "M — Mobility"),
    ("CP_cardio",  "#8E44AD", "CP — Cardiopulmonary"),
    ("SDB_sleep",  "#2E86C1", "SDB — Sleep-disordered breathing"),
]:
    fig.add_trace(go.Scatter(
        x=traj_df["day"], y=traj_df[col],
        mode="lines", line=dict(color=color, width=2), name=name,
    ), row=1, col=1)

# Threshold line
fig.add_hline(y=0.3, line_dash="dash", line_color="red",
              annotation_text="anomaly threshold (0.3)", row=1, col=1)

# Row 2 — hazard rates
days_range = list(range(n_days - 1))
for hazard, color, name in [
    (hazard_M,   "#E67E22", "λ — Mobility"),
    (hazard_CP,  "#8E44AD", "λ — Cardiopulmonary"),
    (hazard_SDB, "#2E86C1", "λ — SDB"),
]:
    fig.add_trace(go.Scatter(
        x=days_range, y=hazard,
        mode="lines", line=dict(color=color, width=1.5), name=name,
        showlegend=False,
    ), row=2, col=1)

fig.update_yaxes(title_text="Severity (0–1)", row=1, col=1)
fig.update_yaxes(title_text="λ (events/day)", row=2, col=1)
fig.update_xaxes(title_text="Day", row=2, col=1)
fig.update_layout(height=650, hovermode="x unified", legend_title="State")
fig.update_xaxes(showgrid=True, gridcolor="lightgray")
fig.update_yaxes(showgrid=True, gridcolor="lightgray")
fig.show()


---
## 6 · Feature Generation

Observation model:
```
x(i,j,d) = theta(i,j) + A_j * z_d + noise
```
- `theta` = personal baseline
- `A_j * z_d` = disease shift × latent severity
- `noise` = drawn from feature distribution family

In [8]:
STATE_MAP = {
    'reduced_mobility':           'M_mobility',
    'heart_failure':              'CP_cardio',
    'COPD':                       'CP_cardio',
    'sleep_disordered_breathing': 'SDB_sleep',
}

def generate_daily_features(traj_df, baseline, resident, disease_params):
    records = []
    for _, day in traj_df.iterrows():
        row = {
            'day':          int(day['day']),
            'M_severity':   round(day['M_mobility'], 3),
            'CP_severity':  round(day['CP_cardio'],  3),
            'SDB_severity': round(day['SDB_sleep'],  3),
        }

        for feat, base_vals in baseline.items():
            theta  = base_vals['theta']
            phi    = base_vals['phi']
            lo, hi = base_vals['bounds']
            total_shift = 0.0

            for condition, severity_level in resident.conditions.items():
                if condition not in disease_params:
                    continue
                if feat not in disease_params[condition]:
                    continue
                params    = disease_params[condition][feat]
                delta     = params['delta_severe'] if severity_level == 'severe' else params['delta_mild']
                z_current = day[STATE_MAP.get(condition, 'M_mobility')]
                total_shift += z_current * delta

            noise = np.random.normal(0, max(phi * 0.5, 0.01))
            value = theta + total_shift + noise
            value = max(lo, value)
            if hi is not None:
                value = min(hi, value)
            if base_vals['family'] == 'beta':
                value = np.clip(value, 0, 1)

            # Integer rounding for count-based features
            if base_vals['family'] in ('negative_binomial', 'zero_inflated_nb'):
                value = max(0, round(value))
                row[feat] = int(value)
            else:
                row[feat] = round(float(value), 2)
        records.append(row)
    return pd.DataFrame(records)


resident = RESIDENT_CONFIGS[1]
synthetic_df = generate_daily_features(traj_df, baseline, resident, DISEASE_PARAMS)

print('Resident:', resident.resident_id)
print('Conditions:', resident.conditions)
print('Shape:', synthetic_df.shape)
synthetic_df.head(8)

Resident: R002_heart_failure
Conditions: {'heart_failure': 'moderate'}
Shape: (180, 22)


,day,M_severity,CP_severity,SDB_severity,chair_duration,daytime_bed_duration,inactive_duration,active_duration,large_motion_count,small_motion_count,...,small_motions_in_bed,large_motions_night,bed_exit_events,sleep_fragmentation_idx,breathing_rate_mean,breathing_rate_var,apnea_count,apnea_duration,cheyne_stokes_duration,night_bathroom_visits
0,0,0.050,0.050,0.050,212.48,58.77,333.52,92.68,12.91,32.85,...,9.99,2.10,1.45,0.15,15.66,1.06,0.02,0.00,0.24,1.16
1,1,0.052,0.052,0.052,164.68,72.61,280.42,84.76,8.22,27.74,...,9.14,2.39,1.32,0.13,14.54,1.20,0.00,0.02,0.27,0.99
2,2,0.057,0.059,0.056,180.60,63.13,238.75,86.64,10.64,24.99,...,8.96,1.71,1.41,0.19,15.28,1.21,0.00,0.01,0.30,1.57
3,3,0.063,0.064,0.063,194.29,75.74,283.94,102.21,12.40,40.38,...,8.30,1.67,0.89,0.15,14.91,1.30,0.00,0.00,0.34,1.99
4,4,0.071,0.070,0.067,189.82,60.57,303.60,95.08,9.95,28.71,...,8.97,1.76,1.93,0.13,13.51,1.24,0.01,0.02,0.35,1.68
5,5,0.074,0.074,0.069,179.13,57.41,326.51,95.42,8.85,37.38,...,8.59,2.25,1.28,0.14,14.11,1.19,0.01,0.01,0.37,1.53
6,6,0.078,0.080,0.076,115.91,77.23,247.81,85.17,9.76,23.53,...,8.52,1.98,1.40,0.16,13.66,1.40,0.00,0.00,0.40,1.91
7,7,0.074,0.084,0.078,199.13,54.77,289.52,85.52,16.15,31.91,...,8.36,1.87,1.07,0.16,14.60,1.35,0.00,0.01,0.42,1.21


### Feature timeseries

In [10]:
feat_cols = ['active_duration', 'night_bathroom_visits', 'bed_exit_events', 'breathing_rate_mean', 'cheyne_stokes_duration']
feat_cols = [f for f in feat_cols if f in synthetic_df.columns]

fig = sp.make_subplots(rows=2, cols=3, subplot_titles=feat_cols)
positions = [(r, c) for r in range(1, 3) for c in range(1, 4)]

for feat, (row, col) in zip(feat_cols, positions):
    fig.add_trace(go.Scatter(
        x=synthetic_df['day'], y=synthetic_df[feat],
        mode='lines', line=dict(width=1.5), name=feat, showlegend=False,
    ), row=row, col=col)

fig.update_layout(
    height=600,
    title_text='Simulated features — ' + resident.resident_id,
    title_font_size=13,
)
fig.update_xaxes(showgrid=True, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridcolor='lightgray')
fig.show()

---
## 7 · Labelled Output

Ground-truth labels are a first-class output.

In [ ]:
def label_day(row, threshold=0.3):
    if row['M_severity']   > threshold: return 'mobility_decline'
    if row['CP_severity']  > threshold: return 'cardiopulmonary_event'
    if row['SDB_severity'] > threshold: return 'sdb_elevated'
    return 'normal'

synthetic_df['ground_truth_label'] = synthetic_df.apply(label_day, axis=1)

print('Label distribution:')
print(synthetic_df['ground_truth_label'].value_counts())

out_file = 'synthetic_' + resident.resident_id + '.csv'
synthetic_df.to_csv(out_file, index=False)
print('Exported:', out_file)
print('Shape:', synthetic_df.shape)